In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


faqs = [
    "What is the estimated delivery time for my order?",
    "Can I cancel my order after placing it?",
    "What is your refund policy for cancelled or failed orders?",
    "Can I substitute an item in my order before it is prepared?",
    "How can I contact customer support?",
    "What should I do if an item is missing from my order?"
]


model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(faqs)
embeddings = np.array(embeddings, dtype=np.float32)


dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Number of FAQs:", len(faqs))
print("Index size:", index.ntotal)

def search_faq(query, k=2):
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding, dtype=np.float32)

    distances, indices = index.search(query_embedding, k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append((faqs[idx], float(dist)))
    return results

queries = [
    "How do I get money back?",
    "My food hasn't arrived yet."
]

for q in queries:
    print("\\n" + "="*70)
    print("Query:", q)
    results = search_faq(q)

    for i, (faq, distance) in enumerate(results, start=1):
        print(f"{i}. FAQ: {faq}")
        print(f"   L2 Distance: {distance:.4f}")


c:\Users\skadi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4824.43it/s]


Number of FAQs: 6
Index size: 6
\n======================================================================
Query: How do I get money back?
1. FAQ: How can I contact customer support?
   L2 Distance: 1.0072
2. FAQ: What is your refund policy for cancelled or failed orders?
   L2 Distance: 1.0291
\n======================================================================
Query: My food hasn't arrived yet.
1. FAQ: What is the estimated delivery time for my order?
   L2 Distance: 1.1270
2. FAQ: What should I do if an item is missing from my order?
   L2 Distance: 1.2698
